# A5: CNN Architecture com Conv2D, MaxPooling, Flatten e Dense

Este notebook demonstra a construção de uma CNN (Convolutional Neural Network) com a arquitetura:

**Input → Conv2D (32 filtros) → MaxPooling → Conv2D (64 filtros) → MaxPooling → Flatten → Dense → Output**

Documentaremos todos os hiperparâmetros e mostraremos exemplos práticos.

## 1. Importar Bibliotecas Necessárias

Vamos importar TensorFlow, Keras e outras bibliotecas essenciais para construir e treinar modelos CNN.

In [9]:
%pip install -q tensorflow

import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import Sequential, layers
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Configurar matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Adicionar src ao path com path absoluto
workspace_path = r'c:\Users\Inteli\Desktop\g01\src'
if workspace_path not in sys.path:
    sys.path.insert(0, workspace_path)

# Force reload de módulos para garantir as mudanças mais recentes
import importlib
if 'models.cnn_builder' in sys.modules:
    del sys.modules['models.cnn_builder']
if 'models.cnn_data_prep' in sys.modules:
    del sys.modules['models.cnn_data_prep']

try:
    from models.cnn_builder import build_cnn_model, get_model_architecture_summary
    from models.cnn_data_prep import prepare_cnn_inputs
    print("cnn_builder importado com sucesso")
    print("cnn_data_prep importado com sucesso")
except ImportError as e:
    print(f"Erro ao importar módulos: {e}")

print(f"\nTensorFlow version: {tf.__version__}")
print(f"GPU disponível: {len(tf.config.list_physical_devices('GPU')) > 0}")

Note: you may need to restart the kernel to use updated packages.
cnn_builder importado com sucesso
cnn_data_prep importado com sucesso

TensorFlow version: 2.20.0
GPU disponível: False


## 2. Definir Parâmetros da Arquitetura CNN

Vamos definir os hiperparâmetros principais para construir nossa arquitetura CNN:

- **input_shape**: Dimensões da entrada (altura, largura, canais)
- **conv1_filters**: Número de filtros na primeira convolução (default: 32)
- **conv2_filters**: Número de filtros na segunda convolução (default: 64)  
- **kernel_size**: Tamanho do kernel/filtro (default: 3×3)
- **dense_units**: Número de neurônios na camada densa (default: 128)
- **dropout_rate**: Taxa de dropout para regularização (default: 0.5)

In [10]:
# ==========================================
# HIPERPARÂMETROS DA CNN - CONFIGURAÇÃO PADRÃO
# ==========================================

# DIMENSÕES DA ENTRADA - Padronizado conforme dados reais
INPUT_HEIGHT = 128           # Altura das imagens (dados ASTER)
INPUT_WIDTH = 128            # Largura das imagens (dados ASTER)
INPUT_CHANNELS = 9           # 9 bandas ASTER (equivalente a RGB multiespectral)
INPUT_SHAPE = (INPUT_HEIGHT, INPUT_WIDTH, INPUT_CHANNELS)

# NÚMERO DE CLASSES - Conforme problema de classificação
N_CLASSES = 2  # Classificação binária

# ==========================================
# PARÂMETROS DAS CAMADAS CONVOLUCIONAIS
# ==========================================

# Primeira convolução
CONV1_FILTERS = 32          # Número de filtros (mais = mais features capturadas)
KERNEL_SIZE_1 = (3, 3)     # Tamanho do kernel (3x3 é padrão, pode usar 5x5, 7x7)

# Segunda convolução
CONV2_FILTERS = 64          # Mais filtros para features mais complexas
KERNEL_SIZE_2 = (3, 3)     # Mesmo tamanho de kernel

# ==========================================
# PARÂMETROS DE REGULARIZAÇÃO
# ==========================================

L2_REGULARIZER = 0.001       # Coeficiente L2
CONV_DROPOUT_RATE = 0.2      # Dropout após convoluções

# ==========================================
# PARÂMETROS DAS CAMADAS DE POOLING
# ==========================================

POOL_SIZE = (2, 2)          # Reduz dimensões pela metade
POOL_STRIDES = 2            # Deslocamento do pool

# ==========================================
# PARÂMETROS DAS CAMADAS DENSAS
# ==========================================

DENSE_UNITS = 128           # Neurônios na camada oculta
DROPOUT_RATE = 0.5          # Taxa de dropout (50%)

# ==========================================
# PARÂMETROS DE TREINAMENTO
# ==========================================

EPOCHS = 50
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2
LEARNING_RATE = 0.001

# Controle de dados
USE_SAMPLE = True           # True = usar amostra para testes rápidos
SAMPLE_SIZE = 295           # Número de amostras
USE_PRECOMPUTED_NORMALIZER = True  # Usar normalizer precomputado

# ==========================================
# CAMINHOS DOS DADOS
# ==========================================

notebook_dir = os.path.dirname(os.path.abspath('notebooks/a05_cnn_teste.ipynb'))
repo_root = Path(workspace_path).parent   # Sobe um nível de src/

DATASET_PATH = repo_root / 'outputs' / 'pixels_dataset.csv'
CODES_PATH = repo_root / 'data' / 'extracted_codes.json'
NORMALIZER_PATH = repo_root / 'outputs' / 'a04_cnn_data_prep' / 'cnn_normalizer_zscore.npz'

# Exibir resumo dos parâmetros
print("="*80)
print("HIPERPARÂMETROS DA ARQUITETURA CNN")
print("="*80)
print(f"\nENTRADA:")
print(f"   Input Shape: {INPUT_SHAPE}")
print(f"   Total de pixels por imagem: {INPUT_HEIGHT * INPUT_WIDTH * INPUT_CHANNELS:,}")
print(f"   Classes: {N_CLASSES}")
print(f"\nCONVOLUÇÃO 1:")
print(f"   Filtros: {CONV1_FILTERS}")
print(f"   Kernel: {KERNEL_SIZE_1}")
print(f"\nCONVOLUÇÃO 2:")
print(f"   Filtros: {CONV2_FILTERS}")
print(f"   Kernel: {KERNEL_SIZE_2}")
print(f"\nREGULARIZAÇÃO (Anti-Overfitting):")
print(f"   L2 Regularizer: {L2_REGULARIZER}")
print(f"   Conv Dropout Rate: {CONV_DROPOUT_RATE * 100}%")
print(f"\nDENSE & DROPOUT:")
print(f"   Dense Units: {DENSE_UNITS}")
print(f"   Dropout Rate: {DROPOUT_RATE * 100}%")
print(f"\nTREINAMENTO:")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Learning Rate: {LEARNING_RATE}")
print(f"\nDADOS:")
print(f"   Dataset: {DATASET_PATH}")
print(f"   Usar amostra: {USE_SAMPLE} ({SAMPLE_SIZE} amostras)")

HIPERPARÂMETROS DA ARQUITETURA CNN

ENTRADA:
   Input Shape: (128, 128, 9)
   Total de pixels por imagem: 147,456
   Classes: 2

CONVOLUÇÃO 1:
   Filtros: 32
   Kernel: (3, 3)

CONVOLUÇÃO 2:
   Filtros: 64
   Kernel: (3, 3)

REGULARIZAÇÃO (Anti-Overfitting):
   L2 Regularizer: 0.001
   Conv Dropout Rate: 20.0%

DENSE & DROPOUT:
   Dense Units: 128
   Dropout Rate: 50.0%

TREINAMENTO:
   Epochs: 50
   Batch Size: 32
   Learning Rate: 0.001

DADOS:
   Dataset: c:\Users\Inteli\Desktop\g01\outputs\pixels_dataset.csv
   Usar amostra: True (295 amostras)


In [11]:
# ==============================================================================
# CARREGAR DADOS REAIS E EXTRAIR LABELS
# ==============================================================================

print("Carregando dataset de pixels...")
print(f"Verificando: {DATASET_PATH}")
print(f"Verificando: {CODES_PATH}")
print(f"Verificando: {NORMALIZER_PATH}")

if DATASET_PATH.exists() and CODES_PATH.exists() and NORMALIZER_PATH.exists():
    print(f"Dataset encontrado: {DATASET_PATH}")
    print(f"Codes file encontrado: {CODES_PATH}")
    print(f"Normalizer path: {NORMALIZER_PATH}")
    
    # Carregar CSV
    df = pd.read_csv(DATASET_PATH)
    if USE_SAMPLE:
        df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42)
    
    print(f"DataFrame carregado com {len(df)} amostras")
    
    # Extrair labels usando a mesma função (sem normalização, só para get labels)
    probe = prepare_cnn_inputs(
        df,
        extracted_codes_path=CODES_PATH,
        normalization='none',
        data_format='channels_last',
        drop_invalid_labels=False,
    )
    
    y_all = probe['y']
    valid_mask = y_all != -1
    df = df.loc[valid_mask].reset_index(drop=True)
    y_train = y_all[valid_mask]
    
    print(f"DataFrame pronto: {len(df)} amostras válidas")
    print(f"Distribuição de classes: {dict(pd.Series(y_train).value_counts().sort_index())}")
    DATA_LOADED = True
else:
    print(f"Erro: Arquivo não encontrado")
    DATA_LOADED = False

# ==============================================================================
# CARREGAR NORMALIZER PRECOMPUTADO
# ==============================================================================

if DATA_LOADED and NORMALIZER_PATH.exists():
    print(f"\nNormalizer encontrado: {NORMALIZER_PATH}")
    
    # Carregar o arquivo .npz
    npz_file = np.load(NORMALIZER_PATH, allow_pickle=True)
    precomputed_normalizer = {
        'method': 'zscore',  # Normalizer foi treinado com zscore
        'mean': npz_file['mean'],
        'std': npz_file['std']
    }
    npz_file.close()
    print("Normalizer carregado com sucesso!")
else:
    precomputed_normalizer = None
    print("Erro ao carregar normalizer precomputado")

Carregando dataset de pixels...
Verificando: c:\Users\Inteli\Desktop\g01\outputs\pixels_dataset.csv
Verificando: c:\Users\Inteli\Desktop\g01\data\extracted_codes.json
Verificando: c:\Users\Inteli\Desktop\g01\outputs\a04_cnn_data_prep\cnn_normalizer_zscore.npz
Dataset encontrado: c:\Users\Inteli\Desktop\g01\outputs\pixels_dataset.csv
Codes file encontrado: c:\Users\Inteli\Desktop\g01\data\extracted_codes.json
Normalizer path: c:\Users\Inteli\Desktop\g01\outputs\a04_cnn_data_prep\cnn_normalizer_zscore.npz
DataFrame carregado com 295 amostras
DataFrame pronto: 295 amostras válidas
Distribuição de classes: {0: np.int64(179), 1: np.int64(116)}

Normalizer encontrado: c:\Users\Inteli\Desktop\g01\outputs\a04_cnn_data_prep\cnn_normalizer_zscore.npz
Normalizer carregado com sucesso!


## 3. Construir Camadas Conv2D com Filtros e Kernels

A primeira camada convolucional extrai features básicas (bordas, texturas) usando 32 filtros 3×3 com ativação ReLU.
A segunda camada convolucional captura features mais complexas usando 64 filtros.

## 2.5 Como Usar Dados Reais (Entrada Customizada)



In [12]:
# ==============================================================================
# PREPARAR ENTRADA DA CNN - prepare_cnn_inputs (conforme cnn_preparacao_dados)
# ==============================================================================

if DATA_LOADED:
    print("\nExecutando pipeline de preparação de dados...")
    
    # Preparar inputs usando a função do módulo (mesmo padrão de cnn_preparacao_dados.ipynb)
    train_out = prepare_cnn_inputs(
        df,
        labels=y_train,
        normalization='zscore',
        normalizer=precomputed_normalizer if USE_PRECOMPUTED_NORMALIZER else None,
        data_format='channels_last',
    )
    
    X_train = train_out['X']
    y_train = train_out['y']
    
    print(f"Pipeline executado com sucesso!")
    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"Classes: {np.unique(y_train)}")
    print(f"Usando normalizer PRÉ-COMPUTADO: {USE_PRECOMPUTED_NORMALIZER}")
else:
    print("Erro: dados não carregados")


Executando pipeline de preparação de dados...
Pipeline executado com sucesso!
X_train shape: (295, 128, 128, 9)
y_train shape: (295,)
Classes: [0 1]
Usando normalizer PRÉ-COMPUTADO: True


In [13]:
# Explicação das camadas Conv2D

print("CAMADA CONV2D - Explicação Técnica\n")
print("="*70)

print("\nPRIMEIRA CONVOLUÇÃO (Conv2D_1):")
print(f"   Filtros: {CONV1_FILTERS}")
print(f"   Kernel: {KERNEL_SIZE_1} (matriz 3x3)")
print(f"   Padding: 'same' (mantém dimensões)")
print(f"   Ativação: ReLU (f(x) = max(0, x))")
print(f"   Parâmetros: 3x3x3x{CONV1_FILTERS} + {CONV1_FILTERS} = {3*3*3*CONV1_FILTERS + CONV1_FILTERS:,}")
print(f"\n   Função: Extrair features básicas (bordas, texturas)")
print(f"   Output shape: ({INPUT_HEIGHT}, {INPUT_WIDTH}, {CONV1_FILTERS})")

print("\n\nSEGUNDA CONVOLUÇÃO (Conv2D_2):")
print(f"   Filtros: {CONV2_FILTERS}")
print(f"   Kernel: {KERNEL_SIZE_2}")
print(f"   Padding: 'same'")
print(f"   Ativação: ReLU")
print(f"   Parâmetros: 3x3x{CONV1_FILTERS}x{CONV2_FILTERS} + {CONV2_FILTERS} = {3*3*CONV1_FILTERS*CONV2_FILTERS + CONV2_FILTERS:,}")
print(f"\n   Função: Extrair features mais complexas")
print(f"   Output shape: (16, 16, {CONV2_FILTERS})  [após pooling]")

print("\n" + "="*70)
print("\nPor que kernel 3x3?")
print("   - Tamanho padrão em deep learning moderno")
print("   - Captura padrões locais eficientemente")
print("   - Menos parâmetros que kernels 5x5 ou 7x7")
print("   - Dois kernels 3x3 = um kernel 5x5 (mesma receptive field)")

print("\nPor que ReLU no lugar de Sigmoid?")
print("   - Convergência mais rápida")
print("   - Evita problema de \"vanishing gradient\"")
print("   - Promove esparsidade (alguns neurônios inativos)")
print("   - Computacionalmente mais eficiente")


CAMADA CONV2D - Explicação Técnica


PRIMEIRA CONVOLUÇÃO (Conv2D_1):
   Filtros: 32
   Kernel: (3, 3) (matriz 3x3)
   Padding: 'same' (mantém dimensões)
   Ativação: ReLU (f(x) = max(0, x))
   Parâmetros: 3x3x3x32 + 32 = 896

   Função: Extrair features básicas (bordas, texturas)
   Output shape: (128, 128, 32)


SEGUNDA CONVOLUÇÃO (Conv2D_2):
   Filtros: 64
   Kernel: (3, 3)
   Padding: 'same'
   Ativação: ReLU
   Parâmetros: 3x3x32x64 + 64 = 18,496

   Função: Extrair features mais complexas
   Output shape: (16, 16, 64)  [após pooling]


Por que kernel 3x3?
   - Tamanho padrão em deep learning moderno
   - Captura padrões locais eficientemente
   - Menos parâmetros que kernels 5x5 ou 7x7
   - Dois kernels 3x3 = um kernel 5x5 (mesma receptive field)

Por que ReLU no lugar de Sigmoid?
   - Convergência mais rápida
   - Evita problema de "vanishing gradient"
   - Promove esparsidade (alguns neurônios inativos)
   - Computacionalmente mais eficiente


## 4. Adicionar Camadas MaxPooling

As camadas MaxPooling2D reduzem as dimensões espaciais pela metade, capturando as features mais importantes.
Isso reduz o número de parâmetros e melhora a robustez das features extraídas.

In [14]:
# Explicação das camadas MaxPooling

print("\n CAMADA MAXPOOLING - Explicação Técnica\n")
print("="*70)

# Calcular dimensões após cada pooling
height_after_pool1 = INPUT_HEIGHT // 2
width_after_pool1 = INPUT_WIDTH // 2
height_after_pool2 = height_after_pool1 // 2
width_after_pool2 = width_after_pool1 // 2

print("\nPRIMEIRA MAXPOOLING (MaxPooling2D_1):")
print(f"   • Pool Size: {POOL_SIZE}")
print(f"   • Strides: {POOL_STRIDES}")
print(f"   • Função: Selecionar o valor máximo em cada janela 2x2")
print(f"   • Input shape: ({INPUT_HEIGHT}, {INPUT_WIDTH}, {CONV1_FILTERS})")
print(f"   • Output shape: ({height_after_pool1}, {width_after_pool1}, {CONV1_FILTERS})")
print(f"   • Redução: 4x menos parâmetros ({(INPUT_HEIGHT*INPUT_WIDTH)/(height_after_pool1*width_after_pool1):.0f}x total)")

print("\nSEGUNDA MAXPOOLING (MaxPooling2D_2):")
print(f"   • Pool Size: {POOL_SIZE}")
print(f"   • Strides: {POOL_STRIDES}")
print(f"   • Input shape: ({height_after_pool1}, {width_after_pool1}, {CONV2_FILTERS})")
print(f"   • Output shape: ({height_after_pool2}, {width_after_pool2}, {CONV2_FILTERS})")

total_pixels_before = INPUT_HEIGHT * INPUT_WIDTH
total_pixels_after = height_after_pool2 * width_after_pool2
reduction_factor = total_pixels_before / total_pixels_after

print(f"\n REDUÇÃO ACUMULATIVA:")
print(f"   • Pixels iniciais: {total_pixels_before:,}")
print(f"   • Pixels finais: {total_pixels_after:,}")
print(f"   • Fator de redução: {reduction_factor:.1f}x")

print("\n" + "="*70)
print("\n Benefícios do MaxPooling:")
print(f"   - Reduz dimensionalidade (de {INPUT_HEIGHT*INPUT_WIDTH:,} para {total_pixels_after:,} pixels)")
print(f"   - Seleção de features mais relevantes (max = mais ativado)")
print(f"   - Aumenta receptive field (visão mais ampla)")
print(f"   - Invariância a pequenas translações")
print(f"   - Regularização implícita (evita overfitting)")
print(f"   - Reduz custo computacional")

# Visualizar exemplo de MaxPooling
print("\n\n EXEMPLO VISUAL - MaxPooling 2x2:")
print("""
Input (4x4):              MaxPooling 2x2:
[1  2  3  4]             [2  4]
[5  6  7  8]      →      [14 16]
[9  10 11 12]
[13 14 15 16]
""")

print("Funcionamento:")
print("   Bloco 1 (superior-esquerdo): max(1,2,5,6) = 6 → 2")
print("   Bloco 2 (superior-direito):  max(3,4,7,8) = 8 → 4")
print("   Bloco 3 (inferior-esquerdo): max(9,10,13,14) = 14 → 14")
print("   Bloco 4 (inferior-direito):  max(11,12,15,16) = 16 → 16")



 CAMADA MAXPOOLING - Explicação Técnica


PRIMEIRA MAXPOOLING (MaxPooling2D_1):
   • Pool Size: (2, 2)
   • Strides: 2
   • Função: Selecionar o valor máximo em cada janela 2x2
   • Input shape: (128, 128, 32)
   • Output shape: (64, 64, 32)
   • Redução: 4x menos parâmetros (4x total)

SEGUNDA MAXPOOLING (MaxPooling2D_2):
   • Pool Size: (2, 2)
   • Strides: 2
   • Input shape: (64, 64, 64)
   • Output shape: (32, 32, 64)

 REDUÇÃO ACUMULATIVA:
   • Pixels iniciais: 16,384
   • Pixels finais: 1,024
   • Fator de redução: 16.0x


 Benefícios do MaxPooling:
   - Reduz dimensionalidade (de 16,384 para 1,024 pixels)
   - Seleção de features mais relevantes (max = mais ativado)
   - Aumenta receptive field (visão mais ampla)
   - Invariância a pequenas translações
   - Regularização implícita (evita overfitting)
   - Reduz custo computacional


 EXEMPLO VISUAL - MaxPooling 2x2:

Input (4x4):              MaxPooling 2x2:
[1  2  3  4]             [2  4]
[5  6  7  8]      →      [14 16]
[9  

## 5. Camadas Flatten e Dense

Após o processamento convolucional e pooling, precisamos achatar (flatten) a representação multidimensional
em um vetor 1D para alimentar as camadas fully connected (Dense).

In [15]:
# Calcular shapes após pooling
reduction_factor = POOL_SIZE[0]  # (2, 2) → fator 2

height_after_pool1 = INPUT_HEIGHT // reduction_factor  # 128 / 2 = 64
width_after_pool1 = INPUT_WIDTH // reduction_factor    # 128 / 2 = 64

height_after_pool2 = height_after_pool1 // reduction_factor  # 64 / 2 = 32
width_after_pool2 = width_after_pool1 // reduction_factor    # 64 / 2 = 32

# Tamanho após flatten
flatten_input_size = height_after_pool2 * width_after_pool2 * CONV2_FILTERS

print("\nCAMADA FLATTEN - Explicação Técnica\n")
print("="*70)

print(f"\nFLATTEN LAYER:")
print(f"   Input shape: ({height_after_pool2}, {width_after_pool2}, {CONV2_FILTERS})")
print(f"   Output shape: ({flatten_input_size:,},)")
print(f"   Operação: Converter tensor 4D em vetor 1D")
print(f"   Parâmetros: 0 (apenas reorganiza dados)")

print("\nCAMADA DENSE - Explicação Técnica\n")
print("="*70)

print(f"\nDENSE LAYER (Oculta):")
print(f"   Input features: {flatten_input_size:,}")
print(f"   Unidades: {DENSE_UNITS}")
print(f"   Ativação: ReLU")
print(f"   L2 Regularization: {L2_REGULARIZER}")
print(f"   Parâmetros: {flatten_input_size} x {DENSE_UNITS} + {DENSE_UNITS} = {flatten_input_size * DENSE_UNITS + DENSE_UNITS:,}")

print(f"\nDROPOUT LAYER:")
print(f"   Taxa: {DROPOUT_RATE * 100}%")
print(f"   Desativa ~{int(DENSE_UNITS * DROPOUT_RATE)} neurônios durante treinamento")
print(f"   Benefício: Reduz overfitting, melhora generalização")

print(f"\nDENSE LAYER (Saída):")
print(f"   Input features: {DENSE_UNITS}")
print(f"   Unidades (classes): {N_CLASSES}")
print(f"   Ativação: Softmax")
print(f"   L2 Regularization: {L2_REGULARIZER}")
print(f"   Parâmetros: {DENSE_UNITS} x {N_CLASSES} + {N_CLASSES} = {DENSE_UNITS * N_CLASSES + N_CLASSES:,}")

# Visualizar transformação de shapes
print("\n\nTRANSFORMAÇÃO DE SHAPES:")
print("="*70)
print(f"Input:              ({INPUT_HEIGHT}, {INPUT_WIDTH}, {INPUT_CHANNELS})")
print(f"After Conv2D_1:     ({INPUT_HEIGHT}, {INPUT_WIDTH}, {CONV1_FILTERS})")
print(f"After MaxPool_1:    ({height_after_pool1}, {width_after_pool1}, {CONV1_FILTERS})")
print(f"After Conv2D_2:     ({height_after_pool1}, {width_after_pool1}, {CONV2_FILTERS})")
print(f"After MaxPool_2:    ({height_after_pool2}, {width_after_pool2}, {CONV2_FILTERS})")
print(f"After Flatten:      ({flatten_input_size:,},)")
print(f"After Dense_hidden: ({DENSE_UNITS},)")
print(f"Output (Softmax):   ({N_CLASSES},)    [distribuição de probs]")


CAMADA FLATTEN - Explicação Técnica


FLATTEN LAYER:
   Input shape: (32, 32, 64)
   Output shape: (65,536,)
   Operação: Converter tensor 4D em vetor 1D
   Parâmetros: 0 (apenas reorganiza dados)

CAMADA DENSE - Explicação Técnica


DENSE LAYER (Oculta):
   Input features: 65,536
   Unidades: 128
   Ativação: ReLU
   L2 Regularization: 0.001
   Parâmetros: 65536 x 128 + 128 = 8,388,736

DROPOUT LAYER:
   Taxa: 50.0%
   Desativa ~64 neurônios durante treinamento
   Benefício: Reduz overfitting, melhora generalização

DENSE LAYER (Saída):
   Input features: 128
   Unidades (classes): 2
   Ativação: Softmax
   L2 Regularization: 0.001
   Parâmetros: 128 x 2 + 2 = 258


TRANSFORMAÇÃO DE SHAPES:
Input:              (128, 128, 9)
After Conv2D_1:     (128, 128, 32)
After MaxPool_1:    (64, 64, 32)
After Conv2D_2:     (64, 64, 64)
After MaxPool_2:    (32, 32, 64)
After Flatten:      (65,536,)
After Dense_hidden: (128,)
Output (Softmax):   (2,)    [distribuição de probs]


## 6. Criar Modelo CNN Completo

Agora vamos criar o modelo CNN completo usando a função `build_cnn_model()` do módulo `cnn_builder.py`.
A arquitetura segue o padrão: Input → Conv2D → MaxPooling → Conv2D → MaxPooling → Flatten → Dense → Output

In [16]:
print("\n CONSTRUINDO MODELO CNN COM REGULARIZAÇÃO...\n")

# Criar o modelo usando a função do cnn_builder com regularização
model = build_cnn_model(
    input_shape=INPUT_SHAPE,
    n_classes=N_CLASSES,
    conv1_filters=CONV1_FILTERS,
    conv2_filters=CONV2_FILTERS,
    kernel_size=(3, 3),
    dense_units=DENSE_UNITS,
    dropout_rate=DROPOUT_RATE,
    l2_regularizer=L2_REGULARIZER,
    conv_dropout_rate=CONV_DROPOUT_RATE
)

# Compilar o modelo
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy' if not hasattr(y_train, 'shape') or len(y_train.shape) == 1 else 'categorical_crossentropy',
    metrics=['accuracy']
)

print("Modelo construído e compilado com sucesso!")


 CONSTRUINDO MODELO CNN COM REGULARIZAÇÃO...

Modelo construído e compilado com sucesso!


## 7. Compilar e Resumir Modelo

Vamos compilar o modelo com Adam optimizer e exibir um resumo completo da arquitetura.

In [17]:
print("\n" + "="*80)
print("RESUMO COMPLETO DO MODELO CNN")
print("="*80 + "\n")

# Exibir model.summary() com mais detalhes
model.summary()

# Obter e exibir informações de arquitetura
print("\n" + "="*80)
print("ESTATÍSTICAS DE PARÂMETROS")
print("="*80 + "\n")

architecture_summary = get_model_architecture_summary(model)

print(f"Total de Parâmetros:       {architecture_summary['total_params']:>15,}")
print(f"Parâmetros Treináveis:     {architecture_summary['trainable_params']:>15,}")
print(f"Parâmetros Não-treináveis: {architecture_summary['non_trainable_params']:>15,}")

# Percentual
trainable_pct = (architecture_summary['trainable_params'] / architecture_summary['total_params']) * 100
print(f"\nTreinável: {trainable_pct:.1f}% dos parâmetros")

# Estimar tamanho em MB
size_mb = (architecture_summary['total_params'] * 4) / (1024 ** 2)  # 4 bytes por float32
print(f"Tamanho estimado (float32): {size_mb:.2f} MB")

# Parâmetros por camada
print(f"\n" + "="*80)
print("PARÂMETROS POR CAMADA")
print("="*80 + "\n")

print(f"{'Camada':<20} {'Tipo':<15} {'Output Shape':<25} {'Parâmetros':>15}")
print("-" * 80)

for layer_info in architecture_summary['layers_info']:
    print(f"{layer_info['name']:<20} "
          f"{layer_info['type']:<15} "
          f"{str(layer_info['output_shape']):<25} "
          f"{layer_info['params']:>15,}")

# Informações de compilação
print(f"\n" + "="*80)
print(" CONFIGURAÇÃO DE COMPILAÇÃO")
print("="*80 + "\n")

print(f"Otimizador:  {model.optimizer.__class__.__name__}")
print(f"Loss:        {model.loss}")
print(f"Métricas:    {', '.join(model.metrics_names) if model.metrics_names else 'Nenhuma'}")

print(f"\n" + "="*80)
print("Modelo CNN pronto para treinamento!")
print("="*80)



RESUMO COMPLETO DO MODELO CNN



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 32)   │         2,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_conv1 (Dropout)         │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpooling2d_1 (MaxPooling2D)   │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_conv2 (Dropout)         │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpooling2d_2 (MaxPooling2D)   │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 65536)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 128)            │     8,388,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_dense (Dropout)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,410,114 (32.08 MB)

 Trainable params: 8,410,114 (32.08 MB)

 Non-trainable params: 0 (0.00 B)


ESTATÍSTICAS DE PARÂMETROS

Total de Parâmetros:             8,410,114
Parâmetros Treináveis:           8,410,114
Parâmetros Não-treináveis:               0

Treinável: 100.0% dos parâmetros
Tamanho estimado (float32): 32.08 MB

PARÂMETROS POR CAMADA

Camada               Tipo            Output Shape                   Parâmetros
--------------------------------------------------------------------------------
conv2d_1             Conv2D          (None, 128, 128, 32)                2,624
dropout_conv1        Dropout         (None, 128, 128, 32)                    0
maxpooling2d_1       MaxPooling2D    (None, 64, 64, 32)                      0
conv2d_2             Conv2D          (None, 64, 64, 64)                 18,496
dropout_conv2        Dropout         (None, 64, 64, 64)                      0
maxpooling2d_2       MaxPooling2D    (None, 32, 32, 64)                      0
flatten              Flatten         (None, 65536)                           0
dense_hidden         Dense        